In [1]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

# torch.set_printoptions(profile='default')
# torch.set_printoptions(profile='full')
torch.set_printoptions(precision=6)


In [2]:
column_headers = ["date","timestamp","open","high","low","close","volume"]
filename = "/mnt/c/Shaukat/code_repo/HighFrequencyTradingCoding/data/DAT_MT_AUDUSD_M1_202411.csv"
df = pd.read_csv(filename, names=column_headers)

# combine dates and timestamps into single datetime column
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['timestamp'], format='%Y.%m.%d %H:%M')

# drop date, timestamp and volume
df.drop(columns=["date", "timestamp", "volume"], inplace=True)

# Setting datetime as index
# df.set_index("datetime", inplace=True)

# Sort dataframe by datetime
df.sort_values(by="datetime", inplace=True)

# Add price label
df['label'] = df.apply(lambda row: 1 if row['close'] > row['open'] else 0, axis=1)

# Reset index
df.reset_index(inplace=True, drop=True)

# Remove Nans
print(f"len_df: {len(df)} before removing NA")
df.dropna(inplace=True)
print(f"len_df: {len(df)} after removing NA")


len_df: 29658 before removing NA
len_df: 29658 after removing NA


In [3]:
# Add this column for debugging
df.reset_index(inplace=True)

In [4]:
df.head(20)

,index,open,high,low,close,datetime,label
0,0,0.65759,0.65767,0.65757,0.65758,2024-11-01 00:00:00,0
1,1,0.65759,0.65759,0.65750,0.65754,2024-11-01 00:01:00,0
2,2,0.65754,0.65764,0.65754,0.65761,2024-11-01 00:02:00,1
3,3,0.65760,0.65760,0.65751,0.65751,2024-11-01 00:03:00,0
4,4,0.65753,0.65758,0.65751,0.65755,2024-11-01 00:04:00,1
5,5,0.65755,0.65761,0.65752,0.65754,2024-11-01 00:05:00,0
6,6,0.65753,0.65760,0.65753,0.65755,2024-11-01 00:06:00,1
7,7,0.65754,0.65757,0.65753,0.65756,2024-11-01 00:07:00,1
8,8,0.65757,0.65757,0.65748,0.65750,2024-11-01 00:08:00,0
9,9,0.65750,0.65751,0.65739,0.65739,2024-11-01 00:09:00,0


# DataLoader

Prepare DATASET and DATALOADER

https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

A custom Dataset class must implement three functions: __init__, __len__, and __getitem__.
        

In [5]:
# Sequence to Sequence Dataset and Data Loader
# Read last five minutes and get the labels of the next five minutes
# x = [x1,x2,x3,x4,x5] where x1 in R4
# y = [y6,y7,y8,y9,y10]

class ForexSeq2Seq(Dataset):
    def __init__(self, csv_files, transform = None, input_window=2, output_window=5):
        # init class parameters
        self.input_window = input_window
        self.output_window = output_window
        self.transform = transform

        dataframes = []
        column_headers = ["date","timestamp","open","high","low","close","volume"]
        for filename in csv_files:
            print(f"reading {filename}")
            df = pd.read_csv(filename, names=column_headers)

            # combine dates and timestamps into single datetime column
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['timestamp'], format='%Y.%m.%d %H:%M')

            # drop date, timestamp and volume
            df.drop(columns=["date", "timestamp", "volume"], inplace=True)

            # Append dataframes
            dataframes.append(df)
        
        # get data
        self.data = pd.concat(dataframes)

        # Remove Nans
        print(f"len_df: {len(self.data)} before removing NA")
        self.data.dropna(inplace=True)
        print(f"len_df: {len(self.data)} after removing NA")

        # Sort dataframe by datetime
        self.data.sort_values(by="datetime", inplace=True)
        self.data.reset_index(drop=True, inplace=True)

        # Add price label
        self.data['label'] = self.data.apply(lambda row: 1 if row['close'] > row['open'] else 0, axis=1)

        # Add this column for debugging
        self.data.reset_index(inplace=True)
        # self.data['datetimestring'] = self.data['datetime'].astype(int)

    def __len__(self):
        '''
        Make sure that we do not access data that do not contain complete input and output
        Given a total dataset of N rows: 
            The first sample starts at index 0 and includes rows up to input_window + output_window - 1.
            The last sample starts at index N - input_window - output_window.
            Thus, the total number of valid samples is: len(self.data) - self.input_window - self.output_window + 1
            Assume:
            N = 10 rows.
            input_window = 3.
            output_window = 2.
            Valid Samples:
            Index	Input Window Rows	Output Window Rows
            0           Rows [0, 1, 2]      Rows [3, 4]
            1	        Rows [1, 2, 3]	    Rows [4, 5]
            2	        Rows [2, 3, 4]	    Rows [5, 6]
            3	        Rows [3, 4, 5]	    Rows [6, 7]
            4	        Rows [4, 5, 6]	    Rows [7, 8]
            5	        Rows [5, 6, 7]	    Rows [8, 9]
        len = N - input_window - output_window + 1 = 10 - 3 - 2 + 1 = 6 i.e. Dataloader will not go beyond 5
        '''
        return len(self.data) - self.input_window - self.output_window + 1
    
    def __getitem__(self, idx):
        '''
        It will be passed integer index internally whose range is determined by __len__
        '''
        input_start = idx
        input_end = idx + self.input_window
        output_start = input_end
        output_end = output_start + self.output_window

        # Get the input sequence 
        input_seq = self.data.iloc[input_start:input_end][["open","high","low","close", "index"]].to_numpy(dtype=float)
        output_seq = self.data.iloc[output_start:output_end][["index", "label"]].to_numpy()


        return torch.tensor(input_seq, dtype=torch.float64), torch.tensor(output_seq, dtype=torch.long) 
        



In [6]:
# Init dataset
list_of_files = ['/mnt/c/Shaukat/code_repo/HighFrequencyTradingCoding/data/DAT_MT_AUDUSD_M1_202411.csv']
input_window = 5 # Read last five minutes
output_window=5 # Output next five minutes

# Read the dataset
dataset = ForexSeq2Seq(csv_files=list_of_files, input_window=input_window, output_window=output_window)

reading /mnt/c/Shaukat/code_repo/HighFrequencyTradingCoding/data/DAT_MT_AUDUSD_M1_202411.csv
len_df: 29658 before removing NA
len_df: 29658 after removing NA


In [7]:
print(type(dataset))
print("-----")
print(dataset.__dict__.keys())
print("-----")
print(dataset.data.head(5))
print("-----")
print(type(dataset.data))
print("-----")
print(dataset.data.dtypes)

<class '__main__.ForexSeq2Seq'>
-----
dict_keys(['input_window', 'output_window', 'transform', 'data'])
-----
   index     open     high      low    close            datetime  label
0      0  0.65759  0.65767  0.65757  0.65758 2024-11-01 00:00:00      0
1      1  0.65759  0.65759  0.65750  0.65754 2024-11-01 00:01:00      0
2      2  0.65754  0.65764  0.65754  0.65761 2024-11-01 00:02:00      1
3      3  0.65760  0.65760  0.65751  0.65751 2024-11-01 00:03:00      0
4      4  0.65753  0.65758  0.65751  0.65755 2024-11-01 00:04:00      1
-----
<class 'pandas.core.frame.DataFrame'>
-----
index                int64
open               float64
high               float64
low                float64
close              float64
datetime    datetime64[ns]
label                int64
dtype: object


In [8]:
# Lets check the dataloader
dataloader = DataLoader(dataset, batch_size=1)

In [15]:
# iterate through dataloader
ex_num = 0
for features, labels in dataloader:
    print(f"------Sequence#: {ex_num}  ------")
    print(f"features: {features}")
    print(f"labels: {labels}")
    print('\n')
    if ex_num == 5:
        break
    ex_num += 1

------Sequence#: 0  ------
features: tensor([[[0.657590, 0.657670, 0.657570, 0.657580, 0.000000],
         [0.657590, 0.657590, 0.657500, 0.657540, 1.000000],
         [0.657540, 0.657640, 0.657540, 0.657610, 2.000000],
         [0.657600, 0.657600, 0.657510, 0.657510, 3.000000],
         [0.657530, 0.657580, 0.657510, 0.657550, 4.000000]]],
       dtype=torch.float64)
labels: tensor([[[5, 0],
         [6, 1],
         [7, 1],
         [8, 0],
         [9, 0]]])


------Sequence#: 1  ------
features: tensor([[[0.657590, 0.657590, 0.657500, 0.657540, 1.000000],
         [0.657540, 0.657640, 0.657540, 0.657610, 2.000000],
         [0.657600, 0.657600, 0.657510, 0.657510, 3.000000],
         [0.657530, 0.657580, 0.657510, 0.657550, 4.000000],
         [0.657550, 0.657610, 0.657520, 0.657540, 5.000000]]],
       dtype=torch.float64)
labels: tensor([[[ 6,  1],
         [ 7,  1],
         [ 8,  0],
         [ 9,  0],
         [10,  0]]])


------Sequence#: 2  ------
features: tensor([[[0.657

In [16]:
features

tensor([[[0.657550, 0.657610, 0.657520, 0.657540, 5.000000],
         [0.657530, 0.657600, 0.657530, 0.657550, 6.000000],
         [0.657540, 0.657570, 0.657530, 0.657560, 7.000000],
         [0.657570, 0.657570, 0.657480, 0.657500, 8.000000],
         [0.657500, 0.657510, 0.657390, 0.657390, 9.000000]]],
       dtype=torch.float64)

In [17]:
labels

tensor([[[10,  0],
         [11,  1],
         [12,  0],
         [13,  1],
         [14,  0]]])